# Kinematic model and steering calibration

Step 5.5. Rolling radius of the slick tires, steering map, and the kinematic
bicycle model checked against low-slip data.

## Rolling radius, slick tires

The caliper gives 60 mm diameter, unloaded. Pushing the car by hand does not
work: disarmed, the drivers short the motors, the rear wheels brake and skid.
Three pushes over 3 m gave 30.4 / 30.5 / 31.3 mm. Skidding only loses wheel
angle, so those are upper bounds.

Driven runs instead: `drive_hold.py 0.2 0.0 14` (1 s ramps, ~2.6 m), distance
read on a tape, lateral drift added with Pythagoras.

In [1]:
import sys
sys.path.append('../scripts')
import numpy as np
import pandas as pd
import bagtools as bt

dfs = bt.zero_time(bt.load('../data/rolling_radius_driven_parquet'))
js, drive = dfs['joint_states'], dfs['drive']

# tape distance and lateral drift per run [mm]
tape = [(2616, 30), (2610, 107), (2608, 100)]

In [2]:
# each run is one burst of /drive messages
t = drive['t'].values
gaps = np.flatnonzero(np.diff(t) > 2.0)
starts = np.r_[t[0], t[gaps + 1]]
ends = np.r_[t[gaps], t[-1]]

rows = []
for t0, t1, (dx, dy) in zip(starts, ends, tape):
    before = bt.window(js, t0, t0 + 0.8)   # armed, speed 0
    after = bt.window(js, t1 - 0.8, t1)    # stopped, disarming
    dl = after['pl'].median() - before['pl'].median()
    dr = after['pr'].median() - before['pr'].median()
    dist = np.hypot(dx, dy) / 1000
    rows.append({'dist [m]': dist, 'left [rad]': dl, 'right [rad]': dr,
                 'r [mm]': 1000 * dist / ((dl + dr) / 2)})

runs = pd.DataFrame(rows).round(3)
runs

,dist [m],left [rad],right [rad],r [mm]
0,2.616,87.460,87.451,29.914
1,2.612,87.403,87.412,29.885
2,2.610,87.422,87.384,29.861


In [3]:
r = runs['r [mm]']
print(f"r = {r.mean():.2f} mm, sd {r.std():.3f} mm over {len(r)} runs")
print(f"vs caliper 30.0: {(r.mean() / 30.0 - 1) * 100:+.1f} %")
print(f"vs stock 34.15:  {(r.mean() / 34.15 - 1) * 100:+.1f} %")

r = 29.89 mm, sd 0.027 mm over 3 runs
vs caliper 30.0: -0.4 %
vs stock 34.15:  -12.5 %


**r = 29.9 mm**, runs within 0.2%. The 0.4% under the caliper is the tire
under load. A driven wheel slips slightly forward and reads a bit low; the
pushed runs bound it from above. The driven value is the one the firmware
needs: it is the radius the wheels have while the motors turn them.

With the stock value in the firmware, every m/s setpoint on slicks ran 12.5%
slow. Wheels still plateau at ~40 rad/s, so v_max on slicks is 1.20 m/s.

### Check after the firmware update

`WHEEL_RADIUS_M` set to 0.0299 and flashed. One more run, `drive_hold.py 0.2 0.0 12`,
IMU recorded too.

In [4]:
R_WHEEL = 0.0299

d = bt.zero_time(bt.load('../data/radius_check_parquet'))
js_c, drive_c = d['joint_states'], d['drive']
t0, t1 = drive_c['t'].iloc[0], drive_c['t'].iloc[-1]

plateau = bt.window(js_c, t0 + 4, t1 - 4)
print(f"wheel speed {plateau['wl'].mean():.3f} / {plateau['wr'].mean():.3f} rad/s, "
      f"setpoint {0.2 / R_WHEEL:.3f}")

before = bt.window(js_c, t0, t0 + 0.8)
after = bt.window(js_c, t1 - 0.8, t1)
angle = (after[['pl', 'pr']].median() - before[['pl', 'pr']].median()).mean()
dist = np.hypot(2568, 137) / 1000
print(f"r = {1000 * dist / angle:.2f} mm")

wheel speed 6.697 / 6.688 rad/s, setpoint 6.689
r = 29.75 mm


Setpoints now come out right on slicks. This run gives 0.5% under the three
above, within what the tape and the lateral correction allow. 0.0299 stays.